# Física BR · Taller: Inducción Electromagnética I — Flujo, Faraday, Lenz y transformador
### Física BR — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* La inducción electromagnética es el principio de generadores,
transformadores y todo el sistema de generación y transmisión de energía eléctrica. Generadores y sensores
se trabajan en el taller **Inducción II**. Cada punto parte
de un problema real e incluye esquema, sliders, botón **"Mostrar solución"** y cuestionario.

**Convenciones:** tres cifras significativas · coma decimal.

**Ecuaciones clave:**
- Flujo magnético:  Φ = B·A·cos θ    (θ entre B y la normal a la superficie)
- Flujo de un campo no uniforme:  Φ = ∫ B·dA   (conductor largo: Φ = (μ₀·I·L/2π)·ln(b/a))
- Ley de Faraday:  fem = −N·(ΔΦ/Δt)    (el signo indica oposición: ley de Lenz)
- fem en barra móvil:  fem = B·L·v
- Ley de Lenz: la corriente inducida se opone al cambio de flujo que la produce
- Transformador ideal:  Vs/Vp = Ns/Np    ·    Is/Ip = Np/Ns

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*

---
## Punto 1 — Flujo magnético (Ing. Eléctrica/Electrónica)  ★★  [CL, UCM]
Una espira de área  A = 0,0200 m²  se coloca en un campo magnético  B = 0,400 T. Se busca el flujo
magnético cuando la espira es perpendicular al campo y cuando forma 60° con él.

**Flujo magnético:**  Φ = B·A·cos θ    (θ entre B y la normal a la superficie)

> 🔧 **Aplicación en ingeniería:** el flujo magnético es la cantidad clave de la inducción: lo que
> importa no es el campo en sí, sino cuánto atraviesa una superficie. Los sensores, generadores y
> transformadores funcionan haciendo variar este flujo.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega a un asistente: *"Explícame por qué el flujo magnético Φ=B·A·cosθ es MÁXIMO cuando la espira es perpendicular al campo (θ=0) y CERO cuando es paralela (θ=90°). Dame la intuición de 'cuántas líneas atraviesan la espira'."* Primero intenta explicarlo tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(9.5,3.5))
    for ax,th,titulo in [(ax1,0,'θ=0°: flujo MÁXIMO'),(ax2,45,'θ=45°: flujo medio'),(ax3,90,'θ=90°: flujo CERO')]:
        # espira (vista de perfil, inclinada según θ)
        import numpy as _np
        thr=_np.radians(th)
        # el área proyectada perpendicular al campo es A·cos(θ)
        w=1.2*_np.cos(thr)
        ax.add_patch(plt.Rectangle((2-w/2,1),max(w,0.05),2,fill=True,fc='#f39c12',ec='#e67e22',alpha=0.4,lw=2))
        # campo (flechas horizontales)
        for y in _np.linspace(1.2,2.8,4):
            ax.annotate('',xy=(3.5,y),xytext=(0.5,y),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=1.3,alpha=0.6))
        ax.text(2,0.5,titulo,ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
        ax.set_xlim(0,4); ax.set_ylim(0,3.6); ax.axis('off')
    plt.suptitle('Flujo magnético Φ = B·A·cosθ (depende del ángulo)',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def flujo(B=0.4, A=0.02, theta=0.0):
    phi=B*A*np.cos(np.radians(theta))
    ths=np.linspace(0,90,100); phis=B*A*np.cos(np.radians(ths))
    plt.figure(figsize=(9,4)); plt.plot(ths,phis*1000,lw=2,color='navy')
    plt.scatter([theta],[phi*1000],color='red',s=60,zorder=5,label=f'θ={theta:.0f}° → Φ={phi*1000:.2f} mWb')
    plt.xlabel('ángulo θ (grados)'); plt.ylabel('flujo Φ (mWb)')
    plt.title('Flujo vs ángulo (Φ=B·A·cosθ)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(flujo,
    B=FloatSlider(value=0.4,min=0.1,max=1,step=0.1,description='B (T)'),
    A=FloatSlider(value=0.02,min=0.005,max=0.05,step=0.005,description='A (m²)',readout_format='.3f'),
    theta=FloatSlider(value=0.0,min=0,max=90,step=10,description='θ (°)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    B=0.4; A=0.02
    _pc(f"Perpendicular (θ=0°): Φ = B·A·cos0° = 0,400·0,0200·1 = {B*A:.3g} Wb = {B*A*1000:.3g} mWb")
    _pc(f"A 60°: Φ = B·A·cos60° = 0,400·0,0200·0,500 = {B*A*0.5:.3g} Wb")
    _pc("El flujo es máximo cuando B es perpendicular a la superficie (θ=0°).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Una espira de 0,0200 m² está perpendicular a un campo de 0,400 T. Encuentre el flujo magnético (en mWb).",8,0.04,"mWb","Φ=B·A=8,00 mWb.", id_preg="Inducc_p1_1", peso=2.0)
quiz_opcion_multiple("El flujo magnético a través de una espira es máximo cuando el campo es:",
    ['Paralelo al plano de la espira', 'Perpendicular a la superficie (θ = 0°)', 'A 45° del plano de la espira', 'Nulo en el centro de la espira'],1,
    "Φ=B·A·cosθ es máximo cuando θ=0° (campo perpendicular a la superficie).", id_preg="Inducc_p1_2", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice que el flujo por una espira es máximo cuando su plano es PARALELO al campo, 'porque el campo la recorre a lo largo'. ¿Correcto o solo convincente?",
    ['Solo convincente: Φ = B·A·cosθ; con el plano paralelo a B ninguna línea lo cruza',
     'Correcto: con el plano paralelo al campo, las líneas recorren toda la superficie de la espira y por eso el flujo es máximo',
     'Correcto: el flujo es máximo cuando θ = 90°, porque ahí el coseno alcanza su valor máximo',
     'Solo convincente: el flujo es el mismo para cualquier orientación, porque el área no cambia'],
    0,
    'El flujo cuenta las líneas que ATRAVIESAN la superficie. Plano paralelo a B: θ = 90° entre B y la normal, cos 90° = 0. El área es la misma, pero su proyección perpendicular a B no.',
    id_preg='Inducc_p1_concep_1', peso=2.0)


---
## Punto 2 — Flujo de un campo NO uniforme: bobina captadora junto a un barraje (Ing. Eléctrica/Subestaciones)  ★★★  [PRC, UCM]
Por un barraje recto y largo de una subestación circula una corriente  I = 800 A  (valor instantáneo). Una espira
rectangular de una bobina captadora está en el mismo plano del conductor, con sus lados de longitud  L = 0,150 m
paralelos a él; el lado cercano está a  a = 2,00 cm  y el lejano a  b = 8,00 cm. Como el campo del conductor
disminuye con la distancia, **Φ = B·A no sirve directamente**: hay que sumar el flujo franja por franja.

**Campo del conductor largo (Ampère):**  B(r) = μ₀·I/(2π·r)      (μ₀ ≈ 4π×10⁻⁷ T·m/A)
**Flujo por franjas de ancho dr:**  dΦ = B(r)·L·dr   ⟹   **Φ = (μ₀·I·L/2π)·ln(b/a)**
**Inductancia mutua:**  M = Φ/I

> 🔧 **Aplicación en ingeniería:** en las subestaciones del Sistema Interconectado Nacional la corriente de los
> barrajes se mide y se vigila con bobinas captadoras y bobinas de Rogowski, que funcionan porque el flujo del
> conductor las atraviesa. El mismo cálculo dice cuánto se acoplan los cables de control y comunicaciones tendidos
> junto a conductores de potencia: el cociente M = Φ/I es el que, con corriente alterna, induce voltajes
> indeseados en ellos (ley de Faraday, Punto 3). Calcular mal el flujo es dimensionar mal esa protección.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que para hallar el flujo de un conductor largo sobre una espira basta con Φ = B·A, usando el campo B del lado de la espira más cercano al conductor. Dame el mejor contraargumento y pregúntame si mi resultado queda por encima o por debajo del real. No me des el valor."* Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    # notación científica con 3 cifras significativas y superíndices
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9.5,3.8))
    # conductor y espira (vista en el plano, cm)
    ax1.plot([0,0],[-1,17],color='#c0392b',lw=5)
    ax1.annotate('',xy=(0,16.5),xytext=(0,12),arrowprops=dict(arrowstyle='-|>',color='#2c3e50',lw=2))
    ax1.text(-0.6,14,'I',fontsize=12,fontweight='bold',color='#2c3e50',ha='right')
    ax1.add_patch(plt.Rectangle((2,0.5),6,15,fill=True,fc='#f39c12',ec='#e67e22',alpha=0.25,lw=2))
    for x in [1.2,2.5,3.5,5,6.5,8,9.5]:
        for y in np.linspace(1.5,14.5,5):
            ax1.text(x,y,'×',ha='center',va='center',fontsize=30/x**0.6+5,color='#2980b9',alpha=0.8)
    ax1.annotate('',xy=(2,-0.6),xytext=(0,-0.6),arrowprops=dict(arrowstyle='<->',color='#7f8c8d'))
    ax1.text(1,-1.8,'a',ha='center',color='#7f8c8d');
    ax1.annotate('',xy=(8,-2.6),xytext=(0,-2.6),arrowprops=dict(arrowstyle='<->',color='#7f8c8d'))
    ax1.text(4,-3.8,'b',ha='center',color='#7f8c8d')
    ax1.text(8.6,8,'L',color='#e67e22',fontsize=11,fontweight='bold')
    ax1.set_xlim(-2,10.5); ax1.set_ylim(-4.5,17.5); ax1.axis('off')
    ax1.set_title('Espira junto a un barraje: B ∝ 1/r',fontsize=10)
    # B(r) y área bajo la curva entre a y b
    I=800.0; a=0.02; b=0.08
    r=np.linspace(0.01,0.10,300); B=2e-7*I/r
    ax2.plot(r*100,B*1e3,color='#2980b9',lw=2)
    rr=np.linspace(a,b,200); ax2.fill_between(rr*100,0,2e-7*I/rr*1e3,color='#f39c12',alpha=0.35,label='∫B dr  (exacto)')
    Bc=2e-7*I/((a+b)/2)
    ax2.plot([a*100,b*100],[Bc*1e3,Bc*1e3],'--',color='#c0392b',lw=2,label='B del centro × ancho (ingenuo)')
    ax2.set_xlabel('distancia al conductor r (cm)'); ax2.set_ylabel('B (mT)')
    ax2.set_title('Φ = L × (área bajo B(r) entre a y b)',fontsize=10); ax2.legend(fontsize=8)
    plt.tight_layout(); plt.show()
esquema()

def flujo_no_uniforme(I=800.0, L_cm=15.0, a_cm=2.0, b_cm=8.0):
    if b_cm <= a_cm:
        print("⛔ Configuración bloqueada: b debe ser mayor que a (el lado lejano no puede estar más cerca que el cercano).")
        return
    L=L_cm/100; a=a_cm/100; b=b_cm/100
    Phi=2e-7*I*L*np.log(b/a); Bc=2e-7*I/((a+b)/2); Phin=Bc*L*(b-a)
    print_es(f"B(a) = {_c3(2e-7*I/a*1e3)} mT   |   B(b) = {_c3(2e-7*I/b*1e3)} mT   |   B(centro) = {_c3(Bc*1e3)} mT")
    print_es(f"Φ exacto = (μ₀·I·L/2π)·ln(b/a) = {_c3(Phi*1e6)} µWb   |   M = Φ/I = {_c3(Phi/I*1e9)} nH")
    print_es(f"Φ ingenuo (B del centro × área) = {_c3(Phin*1e6)} µWb   →   error = {_c3((Phin/Phi-1)*100)} %")
    r=np.linspace(max(a*0.5,0.002),b*1.25,300)
    plt.figure(figsize=(7.5,3.4)); plt.plot(r*100,2e-7*I/r*1e3,color='#2980b9',lw=2)
    rr=np.linspace(a,b,200); plt.fill_between(rr*100,0,2e-7*I/rr*1e3,color='#f39c12',alpha=0.35,label=f'exacto: {Phi*1e6:.1f} µWb'.replace('.',','))
    plt.plot([a*100,b*100],[Bc*1e3]*2,'--',color='#c0392b',lw=2,label=f'ingenuo: {Phin*1e6:.1f} µWb'.replace('.',','))
    plt.xlabel('r (cm)'); plt.ylabel('B (mT)'); plt.legend(fontsize=8)
    plt.title(f'Solo importa b/a = {b/a:.2f}'.replace('.',','),fontsize=10)
    plt.tight_layout(); plt.show()

interact(flujo_no_uniforme,
    I=FloatSlider(value=800.0,min=100,max=2000,step=50,description='I (A)'),
    L_cm=FloatSlider(value=15.0,min=5,max=30,step=1,description='L (cm)'),
    a_cm=FloatSlider(value=2.0,min=0.5,max=10,step=0.5,description='a (cm)'),
    b_cm=FloatSlider(value=8.0,min=1,max=20,step=0.5,description='b (cm)'));

def solucion():
    I=800.0; L=0.150; a=0.0200; b=0.0800
    Ba=2e-7*I/a; Bb=2e-7*I/b
    print_es(f"a) B(a) = 2,00×10⁻⁷·800/0,0200 = {Ba*1e3:.2f} mT ;  B(b) = 2,00×10⁻⁷·800/0,0800 = {Bb*1e3:.2f} mT")
    print_es(f"   El campo en el lado lejano es 4 veces menor: B no es uniforme sobre la espira.")
    Phi=2e-7*I*L*np.log(b/a)
    print_es(f"b) Φ = (μ₀·I·L/2π)·ln(b/a) = 2,00×10⁻⁷·800·0,150·ln(0,0800/0,0200)")
    print_es(f"     = 2,40×10⁻⁵·ln 4 = 2,40×10⁻⁵·{np.log(4):.3f} = {_sci(Phi)} Wb = {Phi*1e6:.1f} µWb")
    Bc=2e-7*I/0.05; Phin=Bc*L*(b-a)
    print_es(f"c) Ingenuo: B(centro, r = 5,00 cm) = {Bc*1e3:.2f} mT ;  Φ_ing = {Bc*1e3:.2f}×10⁻³·(0,150·0,0600) = {_sci(Phin)} Wb")
    print_es(f"   Subestima el flujo en {(1-Phin/Phi)*100:.1f} %: B ∝ 1/r crece rápido cerca del conductor, y la franja")
    print_es(f"   cercana aporta más de lo que el valor del centro 'representa'.")
    print_es(f"d) M = Φ/I = {_sci(Phi)}/800 = {_sci(Phi/I)} H = {Phi/I*1e9:.1f} nH")
    print_es(f"e) Con a = 4,00 cm y b = 16,0 cm: ln(16,0/4,00) = ln 4 → el flujo NO cambia ({_sci(2e-7*I*L*np.log(4))} Wb).")
    print_es(f"   El área se duplica, pero el campo en cada franja correspondiente cae a la mitad.")
    rr=np.linspace(a,b,10001); rm=(rr[1:]+rr[:-1])/2
    Phi_num=np.sum(2e-7*I/rm*L*np.diff(rr))
    print_es(f"Verificación: suma de 10 000 franjas Σ B(r)·L·Δr = {_sci(Phi_num)} Wb ✓ (coincide con la fórmula)")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Campo del barraje en el lado CERCANO de la espira, B(a) = μ₀·I/(2π·a), con I = 800 A y a = 2,00 cm (en mT):",
    8.00,0.03,"mT","B(a)=2,00×10⁻⁷·800/0,0200=8,00×10⁻³ T=8,00 mT.", id_preg="Inducc_p2_1", peso=2.0)
quiz_numerico("Flujo exacto Φ = (μ₀·I·L/2π)·ln(b/a) con L = 0,150 m, a = 2,00 cm y b = 8,00 cm (en µWb):",
    33.3,0.03,"µWb","Φ=2,00×10⁻⁷·800·0,150·ln 4=2,40×10⁻⁵·1,386=3,33×10⁻⁵ Wb=33,3 µWb.", id_preg="Inducc_p2_2", peso=3.0)
quiz_numerico("Inductancia mutua entre el barraje y la espira, M = Φ/I (en nH):",
    41.6,0.04,"nH","M=3,33×10⁻⁵/800=4,16×10⁻⁸ H=41,6 nH. Depende solo de la geometría, no de la corriente.", id_preg="Inducc_p2_3", peso=2.0)
quiz_opcion_multiple("PROBLEMA INESPERADO: se aleja la espira y se ensancha, de modo que a = 4,00 cm y b = 16,0 cm (L igual). ¿Qué pasa con el flujo?",
    ['No cambia: el área se duplica, pero el campo en cada franja cae a la mitad; solo importa b/a',
     'Se reduce a la mitad, porque la espira quedó al doble de distancia del barraje',
     'Se duplica, porque el ancho de la espira, y con él su área, se duplicó',
     'Se reduce a la cuarta parte, porque el campo de un conductor decae como 1/r²'],0,
    "Φ ∝ ln(b/a) y 16,0/4,00 = 8,00/2,00 = 4. El campo de un conductor largo decae como 1/r (no 1/r², que es el de una carga puntual).",
    id_preg="Inducc_p2_4", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante usa Φ = B·A con el campo en el CENTRO de la espira (r = 5,00 cm), obtiene 28,8 µWb y dice: 'es exacto, porque el centro representa el promedio del campo'. ¿Correcto o solo convincente?",
    ['Solo convincente: B ∝ 1/r no es lineal; la franja cercana pesa más y el centro subestima Φ un 13 %',
     'Correcto: el campo en el centro geométrico es, por simetría, el promedio del campo sobre la espira',
     'Correcto: la diferencia con la integral es redondeo y desaparece si se trabaja con más cifras',
     'Solo convincente: el centro SOBREESTIMA el flujo, porque allí el campo es mayor que en el lado lejano'],
    0,
    'El promedio de 1/r entre a y b es ln(b/a)/(b−a), no 1/r_centro; para una curva convexa como 1/r el valor del centro queda POR DEBAJO del promedio. La opción «el centro sobreestima…» acierta el veredicto con la conclusión al revés.',
    id_preg='Inducc_p2_concep_2', peso=2.0)


---
## Punto 3 — Ley de Faraday (Ing. Eléctrica/Generadores)  ★★★  [PRC, UCM]
En una bobina de  N = 50 espiras, el flujo magnético cambia  ΔΦ = 0,00800 Wb  en un tiempo
Δt = 0,0200 s. Se busca la fem inducida.

**Ley de Faraday:**  fem = N·(ΔΦ/Δt)    (en magnitud)

> 🔧 **Aplicación en ingeniería:** la ley de Faraday es el principio de todo generador eléctrico:
> variar el flujo a través de una bobina induce un voltaje. Más espiras y cambios más rápidos generan
> más fem. Es la base de la generación de energía en centrales hidroeléctricas y térmicas (lo verás con datos reales de Hidroituango en el taller Inducción II).

> 🤖 **TIP-IA — Verificador crítico:**
> Dile a un asistente: *"Usé la ley de Faraday fem=−N·ΔΦ/Δt. Actúa como evaluador escéptico: pregúntame qué genera la fem: el valor del flujo o su CAMBIO, y qué pasa si el flujo es grande pero constante. No me des la respuesta."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4))
    t=np.linspace(0,4,100)
    # flujo cambiante (sube)
    flujo=1-np.exp(-t)
    ax1.plot(t,flujo,color='#f39c12',lw=2.5)
    ax1.set_xlabel('tiempo'); ax1.set_ylabel('flujo Φ')
    ax1.set_title('El flujo CAMBIA con el tiempo',fontsize=10); ax1.grid(alpha=0.2,linestyle='--')
    # fem inducida (proporcional a la RAPIDEZ de cambio)
    fem=np.exp(-t)
    ax2.plot(t,fem,color='#c0392b',lw=2.5)
    ax2.fill_between(t,0,fem,color='#c0392b',alpha=0.12)
    ax2.set_xlabel('tiempo'); ax2.set_ylabel('fem inducida')
    ax2.set_title('La fem ∝ rapidez del cambio de flujo',fontsize=10); ax2.grid(alpha=0.2,linestyle='--')
    ax2.annotate('cambio rápido →\nmucha fem',xy=(0.3,0.74),xytext=(1.3,0.7),fontsize=8,color='#2c3e50',
                 arrowprops=dict(arrowstyle='->',color='#7f8c8d'))
    plt.suptitle('Ley de Faraday: fem = −N·ΔΦ/Δt',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def faraday(N=50, dflux=0.008, dt=0.02):
    fem=N*dflux/dt
    dts=np.linspace(0.005,0.1,100); fems=N*dflux/dts
    plt.figure(figsize=(9,4)); plt.plot(dts*1000,fems,lw=2,color='crimson')
    plt.scatter([dt*1000],[fem],color='blue',s=60,zorder=5,label=f'Δt={dt*1000:.0f} ms → fem={fem:.1f} V')
    plt.xlabel('tiempo del cambio Δt (ms)'); plt.ylabel('fem inducida (V)')
    plt.title('Cambio más rápido ⟹ más fem'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(faraday,
    N=FloatSlider(value=50,min=10,max=200,step=10,description='N espiras'),
    dflux=FloatSlider(value=0.008,min=0.002,max=0.02,step=0.002,description='ΔΦ (Wb)',readout_format='.3f'),
    dt=FloatSlider(value=0.02,min=0.005,max=0.1,step=0.005,description='Δt (s)',readout_format='.3f'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    N=50; dflux=0.008; dt=0.02; fem=N*dflux/dt
    _pc(f"Ley de Faraday: fem = N·(ΔΦ/Δt) = 50·(0,00800/0,0200)")
    _pc(f"   = 50·0,400 = {fem:.3g} V")
    _pc("Más espiras o un cambio de flujo más rápido inducen mayor fem.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("En una bobina de 50 espiras el flujo cambia 0,00800 Wb en 0,0200 s. Encuentre la fem inducida (en V).",20,0.04,"V","fem=N·ΔΦ/Δt=20,0 V.", id_preg="Inducc_p3_1", peso=3.0)
quiz_opcion_multiple("Según la ley de Faraday, la fem inducida es mayor si:",
    ['El flujo es grande y se mantiene constante', 'El flujo cambia más rápido o hay más espiras', 'Hay menos espiras, para que la fem no se reparta', 'El campo es nulo justo antes de medir'],1,
    "fem=N·ΔΦ/Δt: más espiras o cambio más rápido ⟹ más fem.", id_preg="Inducc_p3_2", peso=3.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante afirma que un flujo magnético GRANDE a través de una bobina induce una fem grande, aunque el flujo no cambie. ¿Correcto o solo convincente?',
    ['Solo convincente: fem = −N·dΦ/dt; un flujo grande pero constante induce fem cero',
     'Correcto: con más flujo, más líneas atraviesan la bobina y se induce más fem',
     'Correcto: la fem es proporcional al flujo, fem = N·Φ',
     'Solo convincente: la fem depende solo del número de vueltas, no del flujo'],
    0,
    'Faraday mide el RITMO de cambio del flujo, no su valor. Un imán potente quieto junto a una bobina no induce nada; uno débil moviéndose rápido, sí.',
    id_preg='Inducc_p3_concep_3', peso=2.0)


---
## Punto 4 — fem en una barra móvil (Ing. Eléctrica/Transporte)  ★★★  [PRC, ULC]
Una barra conductora de longitud  L = 0,300 m  se mueve a  v = 4,00 m/s  perpendicular a un campo
B = 0,500 T. Se busca la fem generada y la corriente si el circuito tiene  R = 2,00 Ω.

**fem en barra móvil:**  fem = B·L·v    **Corriente:**  I = fem/R

> 🔧 **Aplicación en ingeniería:** una barra que se mueve en un campo genera voltaje: es el principio
> del generador lineal y del freno magnético (usado en trenes y montañas rusas). También explica cómo
> se induce corriente en rieles y estructuras metálicas en movimiento.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele: *"Créame una variante de la barra móvil con otra L, v y B, y pídeme calcular la fem=B·L·v y la corriente si conozco la resistencia. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7,4))
    # rieles conductores
    ax.plot([1,6],[1,1],color='#2c3e50',lw=2)
    ax.plot([1,6],[3.5,3.5],color='#2c3e50',lw=2)
    ax.plot([1,1],[1,3.5],color='#2c3e50',lw=2)
    # campo magnético (saliendo del plano, puntos)
    for x in np.linspace(2,5.5,4):
        for y in np.linspace(1.5,3,3):
            ax.scatter([x],[y],s=40,facecolor='white',edgecolor='#2980b9',linewidth=1.5,zorder=2)
            ax.scatter([x],[y],s=8,color='#2980b9',zorder=3)
    ax.text(5.8,3.2,'B (sale)',color='#2980b9',fontsize=9,fontweight='bold')
    # barra móvil
    ax.plot([4,4],[1,3.5],color='#c0392b',lw=4,zorder=4)
    ax.annotate('',xy=(5,2.25),xytext=(4.2,2.25),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2.5))
    ax.text(4.6,2.6,'v',color='#27ae60',fontsize=11,fontweight='bold')
    ax.text(4,3.8,'barra',ha='center',fontsize=8,color='#c0392b')
    ax.text(3.5,0.4,'Al mover la barra, el área cambia → fem = B·L·v',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0,7); ax.set_ylim(0,4.2); ax.axis('off')
    ax.set_title('fem inducida en una barra móvil')
    plt.tight_layout(); plt.show()
esquema()

def barra(B=0.5, L=0.3, v=4.0, R=2.0):
    fem=B*L*v; I=fem/R
    vs=np.linspace(1,10,100); fems=B*L*vs
    plt.figure(figsize=(9,4)); plt.plot(vs,fems,lw=2,color='navy')
    plt.scatter([v],[fem],color='red',s=60,zorder=5,label=f'v={v:.1f} m/s → fem={fem:.2f} V, I={I:.2f} A')
    plt.xlabel('velocidad v (m/s)'); plt.ylabel('fem inducida (V)')
    plt.title('fem proporcional a la velocidad (fem=B·L·v)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(barra,
    B=FloatSlider(value=0.5,min=0.1,max=1,step=0.1,description='B (T)'),
    L=FloatSlider(value=0.3,min=0.1,max=0.6,step=0.05,description='L (m)'),
    v=FloatSlider(value=4.0,min=1,max=10,step=0.5,description='v (m/s)'),
    R=FloatSlider(value=2.0,min=0.5,max=5,step=0.5,description='R (Ω)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    B=0.5; L=0.3; v=4.0; fem=B*L*v; R=2.0
    _pc(f"fem en barra móvil: fem = B·L·v = 0,500·0,300·4,00 = {fem:.3g} V")
    _pc(f"Corriente: I = fem/R = {fem:.2f}/2,00 = {fem/R:.3g} A")
    _pc("Una barra más larga o más rápida en un campo mayor genera más voltaje.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Una barra de 0,300 m se mueve a 4,00 m/s perpendicular a un campo de 0,500 T. Encuentre la fem generada (en V).",0.6,0.05,"V","fem=B·L·v=0,600 V.", id_preg="Inducc_p4_1", peso=3.0)
quiz_numerico("Si el circuito de esa barra tiene 2,00 Ω, encuentre la corriente inducida (en A).",0.3,0.05,"A","I=fem/R=0,300 A.", id_preg="Inducc_p4_2", peso=3.0)
quiz_opcion_multiple("La fem generada por una barra que se mueve en un campo magnético aumenta si:",
    ['La barra se mueve más lento, dando más tiempo para inducir', 'Aumentan la velocidad, la longitud o el campo', 'El campo es paralelo a la velocidad de la barra', 'La barra se detiene dentro del campo'],1,
    "fem=B·L·v: crece con el campo, la longitud y la velocidad.", id_preg="Inducc_p4_3", peso=3.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: en un campo magnético estático, un estudiante dice que una barra conductora genera fem aunque esté totalmente quieta, porque 'tiene cargas libres'. ¿Correcto o solo convincente?",
    ['Solo convincente: fem = B·L·v exige v ≠ 0; la fuerza q·v×B que separa las cargas necesita velocidad',
     'Correcto: la barra tiene electrones libres, y el campo los empuja hacia los extremos aunque esté quieta',
     'Correcto: la fem depende solo de B y de L, por eso aparece aunque la barra no se mueva',
     'Solo convincente: una barra quieta no genera fem porque es neutra; si estuviera cargada, sí la generaría'],
    0,
    'Un campo magnético estático no ejerce fuerza sobre cargas en reposo (F = q·v×B). Tener carga neta no cambia eso: la clave es la velocidad, no la carga.',
    id_preg='Inducc_p4_concep_4', peso=2.0)


---
## Punto 5 — Ley de Lenz (Ing. Eléctrica/Frenos magnéticos)  ★★  [CL, PRC]
Una bobina de  N = 100 espiras  y área  A = 0,0100 m²  está en un campo que disminuye  ΔB = 0,200 T
en  Δt = 0,100 s. Se busca la magnitud de la fem, y se analiza el sentido de la corriente inducida.

**Faraday con área fija:**  fem = N·A·(ΔB/Δt)    **Lenz:** la corriente se opone al cambio de flujo.

> 🔧 **Aplicación en ingeniería:** la ley de Lenz explica que la corriente inducida siempre se opone
> al cambio que la crea. Esto produce el frenado magnético (sin contacto ni desgaste) usado en trenes,
> montañas rusas y medidores, y es clave para entender pérdidas por corrientes de Foucault.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que la corriente inducida por la ley de Lenz FAVORECE el cambio de flujo que la produce, ayudándolo. Dame el mejor contraargumento."* (Pista: Lenz se OPONE al cambio). Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4))
    ang=np.linspace(0,2*np.pi,100)
    for ax,titulo,acerca in [(ax1,'Imán se ACERCA → la espira lo RECHAZA',True),
                              (ax2,'Imán se ALEJA → la espira lo ATRAE',False)]:
        # espira
        from matplotlib.patches import Ellipse
        ax.add_patch(Ellipse((3,2),1,2.5,fill=False,ec='#c0392b',lw=2.5))
        # imán
        ax.add_patch(plt.Rectangle((0.3,1.6),1,0.8,color='#7f8c8d',alpha=0.8))
        ax.text(0.8,2,'imán',ha='center',va='center',fontsize=8,color='white',fontweight='bold')
        # movimiento del imán
        if acerca:
            ax.annotate('',xy=(2,2),xytext=(1.4,2),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
            ax.text(1.7,2.4,'v',color='#27ae60',fontsize=10,fontweight='bold')
            # fuerza de rechazo (la espira empuja al imán de vuelta)
            ax.annotate('',xy=(1.3,1.3),xytext=(1.9,1.3),arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2))
            ax.text(1.5,0.9,'rechazo',color='#e74c3c',fontsize=8,ha='center')
        else:
            ax.annotate('',xy=(1.4,2),xytext=(2,2),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
            ax.text(1.7,2.4,'v',color='#27ae60',fontsize=10,fontweight='bold')
            ax.annotate('',xy=(1.9,1.3),xytext=(1.3,1.3),arrowprops=dict(arrowstyle='-|>',color='#8e44ad',lw=2))
            ax.text(1.6,0.9,'atracción',color='#8e44ad',fontsize=8,ha='center')
        ax.text(3,0.2,titulo,ha='center',fontsize=8.5,color='#2c3e50',fontweight='bold')
        ax.set_xlim(0,4.5); ax.set_ylim(0,3.8); ax.axis('off')
    plt.suptitle('Ley de Lenz: la corriente inducida SIEMPRE se opone al cambio',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def lenz(N=100, A=0.01, dB=0.2, dt=0.1):
    fem=N*A*dB/dt
    dts=np.linspace(0.02,0.3,100); fems=N*A*dB/dts
    plt.figure(figsize=(9,4)); plt.plot(dts*1000,fems,lw=2,color='crimson')
    plt.scatter([dt*1000],[fem],color='blue',s=60,zorder=5,label=f'Δt={dt*1000:.0f} ms → fem={fem:.2f} V')
    plt.xlabel('tiempo del cambio Δt (ms)'); plt.ylabel('fem inducida (V)')
    plt.title('fem por variación del campo (fem=N·A·ΔB/Δt)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(lenz,
    N=FloatSlider(value=100,min=20,max=300,step=20,description='N espiras'),
    A=FloatSlider(value=0.01,min=0.005,max=0.03,step=0.005,description='A (m²)',readout_format='.3f'),
    dB=FloatSlider(value=0.2,min=0.05,max=0.5,step=0.05,description='ΔB (T)'),
    dt=FloatSlider(value=0.1,min=0.02,max=0.3,step=0.02,description='Δt (s)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    N=100; A=0.01; dB=0.2; dt=0.1; fem=N*A*dB/dt
    _pc(f"fem = N·A·(ΔB/Δt) = 100·0,0100·(0,200/0,100) = 100·0,0100·2,00 = {fem:.3g} V")
    _pc("Ley de Lenz: como el flujo disminuye, la corriente inducida circula para sostenerlo")
    _pc("(se opone al cambio). Ese principio produce el frenado magnético sin contacto.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 5 ---")
quiz_numerico("Una bobina de 100 espiras y 0,0100 m² está en un campo que baja 0,200 T en 0,100 s. Encuentre la fem (en V).",2,0.04,"V","fem=N·A·ΔB/Δt=2,00 V.", id_preg="Inducc_p5_1", peso=2.0)
quiz_opcion_multiple("Según la ley de Lenz, la corriente inducida siempre:",
    ['Refuerza el cambio de flujo que la produce', 'Se opone al cambio de flujo que la produce', 'Es nula si el flujo es grande', 'Va en cualquier sentido, según el material'],1,
    "Lenz: la corriente inducida se opone al cambio que la origina (conservación de energía).", id_preg="Inducc_p5_2", peso=2.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante afirma que la corriente inducida se OPONE al cambio de flujo que la crea, y que si no fuera así se violaría la conservación de la energía. ¿Correcto o solo convincente?',
    ['Correcto: si reforzara el cambio, el sistema se aceleraría solo, creando energía; el signo menos de Faraday es esa oposición',
     'Correcto: la corriente inducida se opone al flujo, y por eso el flujo dentro de la espira siempre termina siendo exactamente cero',
     'Solo convincente: la corriente inducida refuerza el cambio de flujo que la crea',
     'Solo convincente: el sentido de la corriente inducida es arbitrario; solo importa su magnitud'],
    0,
    'Esta vez es CORRECTO. Ojo con la opción «la corriente inducida se opone al…»: Lenz se opone al CAMBIO de flujo, no al flujo; el flujo no se anula, solo se frena su variación. Así funcionan los frenos magnéticos.',
    id_preg='Inducc_p5_concep_5', peso=2.0)


---
## Punto 6 — El transformador (Ing. Eléctrica/Potencia)  ★★★  [PRC, ULC]
Un transformador tiene  Np = 500 vueltas  en el primario y  Ns = 100 vueltas  en el secundario, con
un voltaje de entrada  Vp = 120 V. Se busca el voltaje de salida y la corriente del secundario si el
primario lleva  Ip = 2,00 A.

**Transformador ideal:**  Vs = Vp·(Ns/Np)    ·    Is = Ip·(Np/Ns)   (la potencia se conserva)

> 🔧 **Aplicación en ingeniería:** el transformador cambia el voltaje mediante inducción mutua, y es la
> pieza que hace posible la transmisión de energía a alta tensión (bajas pérdidas) y su uso doméstico
> a baja tensión. Toda la red eléctrica depende de ellos.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega a un asistente: *"Explícame por qué un transformador NO funciona con corriente continua (DC), solo con alterna (AC), a partir de la ley de Faraday. Dame la intuición de por qué se necesita un flujo que cambie."* Primero intenta explicarlo tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7.5,4))
    from matplotlib.patches import Ellipse
    # núcleo (rectángulo)
    ax.add_patch(plt.Rectangle((3,0.8),1,3,fill=False,ec='#7f8c8d',lw=3))
    ax.text(3.5,4,'núcleo',ha='center',fontsize=8,color='#7f8c8d')
    # bobina primaria (pocas espiras, izquierda)
    for y in np.linspace(1.2,3.4,4):
        ax.add_patch(Ellipse((3,y),0.6,0.35,fill=False,ec='#c0392b',lw=2))
    ax.text(1.8,2.3,'primario\nNp espiras',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    ax.text(1.8,1.3,'Vp',ha='center',fontsize=10,color='#c0392b')
    # bobina secundaria (más espiras, derecha)
    for y in np.linspace(1,3.6,7):
        ax.add_patch(Ellipse((4,y),0.6,0.28,fill=False,ec='#2980b9',lw=2))
    ax.text(5.5,2.3,'secundario\nNs espiras',ha='center',fontsize=9,color='#2980b9',fontweight='bold')
    ax.text(5.5,1.3,'Vs',ha='center',fontsize=10,color='#2980b9')
    ax.text(3.5,0.2,'La relación de espiras fija los voltajes: Vs/Vp = Ns/Np',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0.5,6.8); ax.set_ylim(0,4.3); ax.axis('off')
    ax.set_title('El transformador')
    plt.tight_layout(); plt.show()
esquema()

def transformador(Np=500, Ns=100, Vp=120.0, Ip=2.0):
    Vs=Vp*Ns/Np; Is=Ip*Np/Ns
    Nss=np.linspace(50,1000,100); Vss=Vp*Nss/Np
    plt.figure(figsize=(9,4)); plt.plot(Nss,Vss,lw=2,color='navy')
    plt.scatter([Ns],[Vs],color='red',s=60,zorder=5,label=f'Ns={Ns} → Vs={Vs:.1f} V, Is={Is:.1f} A')
    plt.axhline(Vp,color='gray',ls='--',alpha=.5,label=f'Vp={Vp:.0f} V')
    plt.xlabel('vueltas del secundario Ns'); plt.ylabel('voltaje de salida Vs (V)')
    plt.title('El transformador escala el voltaje (Vs=Vp·Ns/Np)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(transformador,
    Np=FloatSlider(value=500,min=100,max=1000,step=50,description='Np'),
    Ns=FloatSlider(value=100,min=50,max=1000,step=50,description='Ns'),
    Vp=FloatSlider(value=120.0,min=12,max=240,step=12,description='Vp (V)'),
    Ip=FloatSlider(value=2.0,min=0.5,max=5,step=0.5,description='Ip (A)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    Np=500; Ns=100; Vp=120.0; Vs=Vp*Ns/Np; Ip=2.0; Is=Ip*Np/Ns
    _pc(f"Voltaje de salida: Vs = Vp·(Ns/Np) = 120·(100/500) = {Vs:.3g} V (reductor)")
    _pc(f"Corriente del secundario: Is = Ip·(Np/Ns) = 2,00·(500/100) = {Is:.3g} A")
    _pc(f"Verificación de potencia: Vp·Ip = {Vp*Ip:.0f} W = Vs·Is = {Vs*Is:.0f} W ✓")
    _pc("Baja el voltaje y sube la corriente en la misma proporción: la potencia se conserva.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 6 ---")
quiz_numerico("Un transformador con Np=500 y Ns=100 recibe Vp=120 V. Encuentre el voltaje de salida Vs (en V).",24,0.04,"V","Vs=Vp·(Ns/Np)=24,0 V.", id_preg="Inducc_p6_1", peso=3.0)
quiz_numerico("Si el primario lleva Ip=2,00 A, encuentre la corriente del secundario Is (en A).",10,0.04,"A","Is=Ip·(Np/Ns)=10,0 A.", id_preg="Inducc_p6_2", peso=3.0)
quiz_opcion_multiple("En un transformador reductor (Ns<Np), respecto al primario el secundario tiene:",
    ['Menor voltaje y menor corriente', 'Menor voltaje y mayor corriente', 'Mayor voltaje y menor corriente', 'Igual voltaje y mayor corriente'],1,
    "Baja el voltaje y sube la corriente proporcionalmente: la potencia se conserva.", id_preg="Inducc_p6_3", peso=3.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante dice que un transformador puede elevar una batería de 12 V DC a 120 V conectándolo directamente. ¿Correcto o solo convincente?',
    ['Solo convincente: sin flujo variable no hay inducción; en DC el secundario no da fem',
     'Correcto: la relación de vueltas eleva cualquier voltaje, sea alterno o continuo',
     'Correcto: con 10 veces más vueltas en el secundario, los 12 V DC se convierten en 120 V DC',
     'Solo convincente: no funciona porque una batería de 12 V entrega muy poca potencia para alimentar un transformador elevador'],
    0,
    'Sin dΦ/dt no hay inducción: V₂/V₁ = N₂/N₁ solo vale en AC. La opción «no funciona porque una batería de…» acierta el veredicto con una razón falsa: el problema no es la potencia, es que la corriente no varía.',
    id_preg='Inducc_p6_concep_6', peso=2.0)


---
## ✅ Fin del taller de Inducción Electromagnética I (Física BR)
Cada punto nació de un problema real: barrajes de subestación, bobinas, frenos magnéticos y transformadores.
Sigue con **Inducción II — Generadores y sensores**: PMSG, el generador de Hidroituango y el sensor VRS. Experimenta con los sliders y
**predice** antes de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=22)
